<a href="https://colab.research.google.com/github/TranGiaHuan0102/VGU_WS26_ClinicalProject_BHTBH/blob/huan%2Fchunking-test/chunking_strategies_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q huggingface_hub langchain_core langchain_huggingface torch tiktoken datasets chonkie[semantic] chonkie chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.8/233.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.4/386.4 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
from huggingface_hub import login


hf_token = ""

# Colab Secrets (🔑 in the sidebar)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass


login(token=hf_token)

In [4]:
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '6aad11231339b5a189922a50', 'name': 'choisongjun', 'fullname': 'HT', 'isPro': False, 'avatarUrl': '/avatars/481c7e7c8d0b9c678adf6db8ab7a351d.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'HuggingFace-CreateDataset', 'role': 'fineGrained', 'createdAt': '2026-10-08T04:12:35.840Z', 'fineGrained': {'canReadGatedRepos': True, 'global': ['discussion.write'], 'scoped': [{'entity': {'_id': '6aad11231339b5a189922a50', 'type': 'user', 'name': 'choisongjun'}, 'permissions': ['repo.contribute.write', 'repo.content.read', 'repo.access.read', 'repo.write', 'discussion.write']}]}}}}


In [5]:
import pandas as pd
from datasets import load_dataset

repo_id = "choisongjun/Tropical_Diseases_Repo_1.3"

dataset_dict = load_dataset(repo_id, token=hf_token)
train_df = dataset_dict["train"].to_pandas()
test_df = dataset_dict["test"].to_pandas()

DISEASE_LABELS = sorted(train_df["disease"].unique())

print(f"train cases: {len(train_df)} | test cases: {len(test_df)} | diseases: {len(DISEASE_LABELS)}")
train_df.head()

README.md:   0%|          | 0.00/478 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 4.58MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  669kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/83 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12 [00:00<?, ? examples/s]

train cases: 83 | test cases: 12 | diseases: 8


,case_id,case_text,disease,images
0,PMC10870101_01,A 51-year-old female patient presented to our ...,dengue,[{'bytes': b'RIFF\xa4\x0e\x00\x00WEBPVP8 \x98\...
1,PMC6417330_01,A 24-year-old female underwent transcranial su...,dengue,[{'bytes': b'RIFFBM\x00\x00WEBPVP8 6M\x00\x00\...
2,PMC10876100_02,"A 43-year-old man, a resident of Uttar Pradesh...",dengue,[{'bytes': b'RIFF`7\x00\x00WEBPVP8 T7\x00\x00\...
3,PMC10932498_01,A 17 year old male patient with no previous co...,dengue,[{'bytes': b'RIFF\xe85\x00\x00WEBPVP8 \xdc5\x0...
4,PMC10062082_01,A 28-year-old male presented with complaint of...,dengue,[{'bytes': b'RIFF\xe6\x1b\x00\x00WEBPVP8 \xda\...


Define chunkers

In [6]:
from chonkie import (
    SemanticChunker,
    RecursiveChunker,
    RecursiveRules,
    RecursiveLevel,
    OverlapRefinery,
)

import re

# ---------- Chunkers ----------
rules = RecursiveRules(levels=[
    RecursiveLevel(delimiters=["\n\n"], include_delim="prev"),
    RecursiveLevel(delimiters=["\n"], include_delim="prev"),
    RecursiveLevel(delimiters=[". ", "! ", "? "], include_delim="prev"),
    RecursiveLevel(whitespace=True),
])

semantic_chunker = SemanticChunker(
    threshold=0.7,
    chunk_size=400,              # tokens
    similarity_window=3,
    min_sentences_per_chunk=2,
    skip_window=0,
)

recursive_chunker = RecursiveChunker(
    tokenizer="character",
    chunk_size=700,              # characters
    rules=rules,
)

overlap_refinery = OverlapRefinery(
    tokenizer="character",
    context_size=120,            # fixed overlap in characters
    mode="recursive",
    rules=rules,
    method="prefix",
    merge=True,
)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Define chunk functions

In [7]:
def clean_text(text):
    # Stop "et al." from being treated as a sentence ending
    text = text.replace("et al.", "et al")
    # Remove image tags
    text = re.sub(r"\[?(?:<[^>]+\.webp>)+(?:-\d+)?\]?", "", text)
    return text

# ---------- Chunk functions----------
def perform_semantic_chunking(text):
    chunks = semantic_chunker.chunk(clean_text(text))
    return [c.text for c in overlap_refinery(chunks)]


def perform_structured_chunking(text):
    chunks = recursive_chunker.chunk(clean_text(text))
    return [c.text for c in overlap_refinery(chunks)]

Define vector database

In [8]:
import chromadb
client = chromadb.PersistentClient(path="/content/drive/MyDrive/colab_projects/VGU_WS26_Project/chroma_db")

In [18]:
def build_collection(records, name, rebuild=False):
    # records: list of {"document": str, "metadata": dict}
    if rebuild:
        try:
            client.delete_collection(name)
        except Exception:
            pass

    collection = client.get_or_create_collection(name)

    batch = client.get_max_batch_size()
    for start in range(0, len(records), batch):
        part = records[start:start + batch]
        collection.upsert(
            ids=[f"chunk_{start + i}" for i in range(len(part))],
            documents=[r["document"] for r in part],
            metadatas=[r["metadata"] for r in part],
        )

    print(f"{name}: built {collection.count()} chunks")
    return collection

Build records for ChromaDB

In [19]:
def make_records(df, chunk_fn):
    records = []
    for row in df.itertuples():
        chunks = chunk_fn(row.case_text)
        for chunk_id, chunk in enumerate(chunks):
            records.append({
                "document": chunk,
                "metadata": {
                    "disease": row.disease,
                    "chunk_id": chunk_id,
                    "total_chunks": len(chunks),
                    "chunk_char": len(chunk),
                },
            })
    return records

semantic_records = make_records(train_df, perform_semantic_chunking)
structured_records = make_records(train_df, perform_structured_chunking)



In [20]:
semantic_collection = build_collection(semantic_records, "semantic_chunking", rebuild=True)

semantic_chunking: built 599 chunks
